# 🧪 06 - Describing Data Lab: Palmer Penguins

A research team has measured 344 penguins on three islands in Antarctica and sent you `penguins.csv`. They want a **numeric description** of the data before anyone draws a single chart:

- What's in the file, and what does each column mean?
- What's typical for each measurement, and how much does it vary?
- Are there outliers, and are they mistakes?
- How do the three species differ?
- Which measurements are related, and does that hold up within each species?

You'll answer with everything from notebooks 01–05, and save a small report. In **Module 08's lab**, you'll turn these same findings into figures, so keep your answers. You'll check them against the charts.

## ✅ Learning Goals
- Run a full descriptive workflow, from raw file to saved summary tables
- Choose statistics that match each column's type and shape
- Separate real outliers from errors, overall and within groups
- Compare groups fairly, and test whether a relationship survives splitting by group

```text
Load → Inspect → Clean (Module 06) → Data dictionary (01)
     → Center & spread (02) → Shape & outliers (03)
     → Compare groups (04) → Relationships (05) → Export the report
```

## 0. Setup
Everything this lab saves goes into **`06-OutputFiles`**. Upload that folder along with this notebook.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

## 1. Load and Look (Change Nothing Yet)

In [ ]:
penguins = pd.read_csv('penguins.csv')
print(penguins.shape)
penguins.head()

In [ ]:
penguins.info()

In [ ]:
penguins.isna().sum()

**One row = one penguin.** Four measurements are each missing twice, and `sex` is missing 11 times. Are the measurement gaps in the same two rows?

In [ ]:
penguins[penguins[MEASUREMENTS].isna().any(axis=1)]

Yes: two penguins with **no measurements at all**. They can't contribute anything to a description of measurements.

## 2. Clean (Module 06)
1. Drop the two rows with no measurements.
2. Keep penguins with unknown `sex`: their measurements are still useful. Just remember they'll drop out of any by-sex comparison.
3. Tidy `sex` from `MALE`/`FEMALE` to `Male`/`Female`.
4. Record meaning with dtypes (notebook 01): `species`, `island`, and `sex` are nominal → `category`.

In [ ]:
clean = penguins.dropna(subset=MEASUREMENTS).copy()
clean['sex'] = clean['sex'].str.title()

for col in ['species', 'island', 'sex']:
    clean[col] = clean[col].astype('category')

print('Rows before:', len(penguins), '| after:', len(clean))
print(clean['sex'].value_counts(dropna=False))
clean.dtypes

## 3. Data Dictionary (Notebook 01)

In [ ]:
data_dictionary = pd.DataFrame({
    'column': clean.columns,
    'meaning': ['Penguin species', 'Island where measured', 'Bill (beak) length',
                'Bill (beak) depth', 'Flipper length', 'Body mass', 'Sex'],
    'kind': ['qualitative', 'qualitative', 'quantitative', 'quantitative',
             'quantitative', 'quantitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'ratio', 'ratio', 'ratio', 'ratio', 'nominal'],
    'dtype': clean.dtypes.astype(str).values,
    'missing': clean.isna().sum().values,
    'distinct': clean.nunique().values,
})
data_dictionary.to_csv(OUT / 'penguins_data_dictionary.csv', index=False)
data_dictionary

## 4. Center and Spread (Notebook 02)
One table with every statistic from notebooks 02 and 03, one row per measurement:

In [ ]:
overall = clean[MEASUREMENTS].describe().T
overall['IQR'] = overall['75%'] - overall['25%']
overall['skew'] = clean[MEASUREMENTS].skew()
overall.round(2)

All four measurements have a mean close to their median, and |skew| below 0.5: no strong lopsidedness like Titanic's fares. Mean and std are reasonable summaries here.

But "roughly symmetric overall" can hide something. This file mixes **three species** of very different sizes. Keep that in mind for section 6.

## 5. Shape and Outliers (Notebook 03)
Reuse the fence function from notebook 03, and count outliers in each measurement:

In [ ]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


for col in MEASUREMENTS:
    low, high = iqr_fences(clean[col])
    n_out = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f'{col:<18} fences {low:8.1f} to {high:8.1f}   outliers: {n_out}')

No outliers at all across the whole dataset. Now ask notebook 03's Practice 3 question: is a value unusual **for its own species**? Compute the fences within each species with `groupby().transform()`, which lines each group's statistic up with the original rows:

In [ ]:
def outliers_within(df, col, group):
    """Boolean mask: is each value outside its own group's IQR fences?"""
    q1 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)


within = pd.DataFrame({col: outliers_within(clean, col, 'species') for col in MEASUREMENTS})
within.sum()

In [ ]:
clean[within.any(axis=1)].sort_values('species')

A few penguins are unusual **for their species**, even though nothing is unusual overall. Look at the rows: are any values impossible? (A small Chinstrap is still a possible penguin. A 30 g penguin, or a 3-meter flipper, would not be.) These look like real, unusual animals, so we **keep** them.

> 🧠 Whether a value is an outlier depends on **which group you compare it to**.

## 6. Comparing Species (Notebook 04)

In [ ]:
species_summary = (
    clean
    .groupby('species', observed=True)
    .agg(
        penguins=('body_mass_g', 'size'),
        mass_mean=('body_mass_g', 'mean'),
        mass_median=('body_mass_g', 'median'),
        mass_std=('body_mass_g', 'std'),
        flipper_mean=('flipper_length_mm', 'mean'),
        bill_length_mean=('bill_length_mm', 'mean'),
        bill_depth_mean=('bill_depth_mm', 'mean'),
    )
    .round(1)
)
species_summary.to_csv(OUT / 'species_summary.csv')
species_summary

Gentoo penguins are much heavier, with longer flippers, but **shallower** bills. Hold on to that "but". It matters in section 7.

### Where Each Species Lives

In [ ]:
pd.crosstab(clean['species'], clean['island'], margins=True)

Chinstraps were measured only on Dream, and Gentoos only on Biscoe. So "Biscoe penguins are heavy" would really be a statement about **Gentoos**: island and species are tangled together (a confounder, notebook 04). Compare islands only **within** a species that lives on more than one (Adelie).

### Species × Sex

In [ ]:
mass_by_species_sex = clean.pivot_table(
    values='body_mass_g', index='species', columns='sex', aggfunc='mean', observed=True,
).round(0)
mass_by_species_sex['difference'] = mass_by_species_sex['Male'] - mass_by_species_sex['Female']
mass_by_species_sex

Within every species, males are heavier. The penguins with unknown sex aren't in this table (section 2).

## 7. Relationships (Notebook 05)

In [ ]:
corr = clean[MEASUREMENTS].corr()
corr.round(2)

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False).round(3)

Flipper length and body mass are very strongly related (bigger penguins, bigger flippers). Now look at **bill length vs. bill depth**: a weak **negative** correlation. Longer bills are shallower?

Check whether that holds up **within each species**:

In [ ]:
rows = []
for species, group in clean.groupby('species', observed=True):
    rows.append({
        'species': species,
        'n': len(group),
        'r_bill_length_depth': group['bill_length_mm'].corr(group['bill_depth_mm']),
    })
rows.append({'species': 'ALL (combined)', 'n': len(clean),
             'r_bill_length_depth': clean['bill_length_mm'].corr(clean['bill_depth_mm'])})

bill_corr = pd.DataFrame(rows).round(3)
bill_corr.to_csv(OUT / 'bill_correlation_by_species.csv', index=False)
bill_corr

🤯 **Within every species, the correlation is positive, and moderate to strong.** Combined, it's negative.

This is **Simpson's paradox**: a relationship that reverses when you combine groups. The species summary in section 6 explains how. Gentoos have long bills **and** shallow bills compared with the others. Pooling a "long and shallow" group with "shorter and deeper" groups creates a downward trend that doesn't exist inside any one species. `species` is the confounder.

You found this with numbers alone, but you had to know to split by species. In Module 08's lab, one scatter plot colored by species shows it instantly, with no need to know ahead of time.

## 8. Export the Report

In [ ]:
clean.to_csv(OUT / 'cleaned_penguins.csv', index=False)
overall.round(2).to_csv(OUT / 'measurement_summary.csv')
corr.round(3).to_csv(OUT / 'correlation_matrix.csv')

print('Saved:', sorted(p.name for p in OUT.glob('*.csv')))

## ❓ Lab Questions
Answer each in the Markdown cell below, with numbers from your output:
1. What does one row represent? How many rows did you remove, and why? Why did you **keep** penguins with unknown sex?
2. Which measurement varies the most **relative to its mean**? (Hint: divide `std` by `mean`.)
3. Why were there no outliers overall, but some within species? Did you keep or remove them, and why?
4. Which species is heaviest, and by how much (mean and median) compared with the lightest?
5. Why can't you compare islands directly? Which comparison would be fair?
6. Explain the bill length vs. bill depth result in your own words, as if to someone who has never heard of Simpson's paradox.

*Your answers here.*

## 🏋️ Practice

### Practice 1 — Species Spread (easy)
For each species, compute the IQR of `flipper_length_mm`. Which species is the most variable, and is it also the one with the largest std?

In [ ]:
# Your code here

### Practice 2 — A Fair Island Comparison (medium)
Filter to **Adelie** penguins only (the one species on all three islands). Compare their body mass across islands with size, mean, median, and std. Is there a real island difference among Adelies?

In [ ]:
# Your code here

### Practice 3 — Hunt for Another Reversal (harder)
Loop over **every pair** of measurements. For each pair, compute the combined correlation and the three within-species correlations. Flag any pair where the combined sign differs from **all three** species' signs. Is bill length/depth the only one?

In [ ]:
# Your code here

## ✏️ Challenge Task
Build a **sex comparison report**: for each species, the number of males and females, and the male − female difference in mean `body_mass_g`, `flipper_length_mm`, and `bill_depth_mm`. Save it as `OUT / 'sex_differences_by_species.csv'`.

In [ ]:
# Your code here

## 🔥 Challenge (Optional) — Same Workflow, New Data
Run sections 1–7 on `pokemon.csv` in a compact form: data dictionary, measurement summary with skew, outliers overall and within `Legendary`, a group comparison by `Generation`, and the strongest correlations. Save at least one table to `OUT` with a `pokemon_` prefix. Write three findings in a Markdown cell.

In [ ]:
# Your code here

*Your three findings here.*

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
### 📝 Summary
This lab described a dataset from scratch, with numbers only:
- **Inspected** first, removed only rows with nothing to contribute, and kept partial rows
- **Recorded meaning** with a data dictionary and `category` dtypes
- **Summarized** center, spread, and shape, then checked outliers overall **and within groups**
- **Compared species** fairly, and spotted island as a stand-in for species
- **Tested a relationship within groups** and found Simpson's paradox

🎓 That's Module 07. **Module 08** picks up right here: the same questions, answered with charts. Its first notebook starts with the Anscombe datasets from notebook 05, and its lab returns to these penguins.